# A professional prediction workflow with `Pipeline`

**Thursday 1 October · 12:30–15:00 · work in pairs**

**Question.** After qualifying, which drivers might finish in the top ten? This is an **invented teaching dataset**, not Formula 1 results. It contains 18 race weekends and 20 fictional drivers per weekend (360 rows). One row represents one driver in one weekend. There are missing numbers, a missing category, and a new circuit category later in time.

**Why this notebook?** In a manual workflow you must remember to fit each imputer, encoder and scaler on the right rows, then apply the same fitted objects in exactly the same order later. A `Pipeline` packages those steps with the model. `fit(X_train, y_train)` learns them together; `predict(X_later)` reuses them. It also lets a parameter search refit the **whole procedure** inside each temporal fold. A pipeline does **not** decide which rows are training or whether a feature is available at prediction time.

**By 15:00, hand in:** one supplied Thursday worksheet per pair and one individual exit ticket. Use short labels in English or Spanish. Predict before running each marked cell. If execution fails, read the saved outputs and explain them.

**Route:** 1 inspect and split → 2 build → 3 fit and inspect → 4 validate → 5 compare temporal folds → 6 apply to new rows → 7 make one controlled change and diagnose limits. Work through the stop points on your worksheet.

## 1. Inspect the information and choose the split

Available **after qualifying, before the race**: qualifying position, practice time gap, previous top-ten rate, grid penalty, circuit type and rain forecast. The target `top10` is the race outcome; it is **never an input feature**. `driver_id` and `week` identify rows and control the split; they are not model features.

**Stop 1.** Predict how many rows belong to training (weeks 1–10), validation (11–14), and the reserved test (15–18). Why must we split before fitting missing-value rules?

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV

path = Path("w04_race_weekends_synthetic_v3.csv")
if not path.exists():
    raise FileNotFoundError("Open the notebook in the folder containing its CSV file.")
data = pd.read_csv(path)
print("Rows, columns:", data.shape)
print(data.head(5).to_string(index=False))
print("The displayed rows are from week 1; later outcomes are not inspected here.")

Rows, columns: (360, 9)
 week driver_id  qualifying_position  practice_gap_s  previous_top10_rate  grid_penalty circuit_type rain_forecast  top10
    1       D01                    4           0.302                0.355             0       street            no      1
    1       D02                    3           0.168                0.448             0       street            no      1
    1       D03                   13           0.981                0.676             0       street            no      0
    1       D04                    6           0.381                0.553             0       street            no      1
    1       D05                    1           0.020                0.630             5       street            no      1
The displayed rows are from week 1; later outcomes are not inspected here.


In [2]:
train = data.loc[data.week <= 10].copy()
valid = data.loc[data.week.between(11, 14)].copy()
test_reserved = data.loc[data.week.between(15, 18)].copy()  # Keep sealed today.
numeric = ["qualifying_position", "practice_gap_s", "previous_top10_rate", "grid_penalty"]
categorical = ["circuit_type", "rain_forecast"]
features = numeric + categorical
X_train, y_train = train[features], train.top10
X_valid, y_valid = valid[features], valid.top10
print("Training / validation / reserved test:", len(train), len(valid), len(test_reserved))
print("Validation circuits:", sorted(valid.circuit_type.unique()))
print("Training circuits:", sorted(train.circuit_type.unique()))
print("Missing training inputs:", train[features].isna().sum().to_dict())
print("Top-ten outcomes in each training weekend:", train.groupby("week").top10.sum().unique().tolist())
assert train.week.max() < valid.week.min() < test_reserved.week.min()
assert "top10" not in features and "week" not in features

Training / validation / reserved test: 200 80 80
Validation circuits: ['high_speed', 'road', 'street', 'technical']
Training circuits: ['high_speed', 'road', 'street']
Missing training inputs: {'qualifying_position': 0, 'practice_gap_s': 17, 'previous_top10_rate': 13, 'grid_penalty': 0, 'circuit_type': 0, 'rain_forecast': 14}
Top-ten outcomes in each training weekend: [10]


## 2. Build one estimator from several steps

The **numeric branch** replaces missing values with the **training median** and standardises each feature using training statistics. The **categorical branch** replaces a missing category with the most frequent training category and one-hot encodes it. `handle_unknown="ignore"` allows a later `technical` circuit even though none appears in training. Both branches feed a logistic regression classifier.

`ColumnTransformer` runs different preprocessing on different columns; the outer `Pipeline` joins preprocessing and model. The names `prep`, `num`, `cat` and `model` let us inspect and tune steps. `__` reaches into a named step, such as `model__C`.

**Stop 2.** Which operation actually learns medians, scales, categories and model coefficients: constructing the Python objects or calling `fit`?

In [3]:
numeric_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical),
])
model = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
print(model)
print("Example tunable parameter:", model.get_params()["model__C"])
print("No fitting has happened yet.")

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('fill',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['qualifying_position',
                                                   'practice_gap_s',
                                                   'previous_top10_rate',
                                                   'grid_penalty']),
                                                 ('cat',
                                                  Pipeline(steps=[('fill',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                  

## 3. Fit on training only, then inspect what was learned

This is the only `fit` call for the first candidate. The fitted preprocessing lives inside `model.named_steps["prep"]`. In particular, `statistics_` gives the four learned numeric medians in the order of `numeric`.

**Stop 3.** Predict whether a validation row can change those stored medians when we call `predict`.

In [4]:
model.fit(X_train, y_train)
learned = model.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_.copy()
print("Training medians:", dict(zip(numeric, learned.round(3))))
print("Transformed feature count:", len(model.named_steps["prep"].get_feature_names_out()))
print("First eight transformed names:", model.named_steps["prep"].get_feature_names_out()[:8].tolist())
assert len(learned) == len(numeric)
assert not any(pd.isna(learned))

Training medians: {'qualifying_position': np.float64(10.5), 'practice_gap_s': np.float64(0.695), 'previous_top10_rate': np.float64(0.493), 'grid_penalty': np.float64(0.0)}
Transformed feature count: 9
First eight transformed names: ['num__qualifying_position', 'num__practice_gap_s', 'num__previous_top10_rate', 'num__grid_penalty', 'cat__circuit_type_high_speed', 'cat__circuit_type_road', 'cat__circuit_type_street', 'cat__rain_forecast_no']


## 4. Validate the fitted procedure

A **baseline** is a fair simple comparison: it always predicts the most frequent class learned from training. We report accuracy and balanced accuracy because class proportions can make accuracy misleading. The validation period helps us compare approaches. The reserved test stays unopened.

`model.predict(X_valid)` performs the fitted preprocessing and classifier prediction in one call. We do not manually impute or encode validation.

**Stop 4.** Before running, identify the line that would be unsafe: `preprocess.fit_transform(data[features])` before the time split, or `model.predict(X_valid)` after fitting on training? Explain in one sentence.

In [5]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_pred = baseline.predict(X_valid)
valid_pred = model.predict(X_valid)
print("Baseline accuracy / balanced accuracy:", round(accuracy_score(y_valid, baseline_pred),3), round(balanced_accuracy_score(y_valid, baseline_pred),3))
print("Pipeline accuracy / balanced accuracy:", round(accuracy_score(y_valid, valid_pred),3), round(balanced_accuracy_score(y_valid, valid_pred),3))
print("Stored medians unchanged after predict:", bool((learned == model.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()))
assert (learned == model.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()

Baseline accuracy / balanced accuracy: 0.5 0.5
Pipeline accuracy / balanced accuracy: 0.863 0.863
Stored medians unchanged after predict: True


## 5. The professional advantage: tune the whole procedure in temporal folds

Within the **training period only**, compare `model__C` values 0.1, 1 and 10. Two expanding folds are explicit: weeks 1–6 → 7–8, then weeks 1–8 → 9–10. In each fold, `GridSearchCV` clones the **entire pipeline** and fits its imputer, encoder, scaler and model again on that fold's earlier rows. A manual workflow can easily fit preprocessing once on all weeks 1–10 before cross-validation, leaking later-fold information.

The winning pipeline is refitted on weeks 1–10. We may then inspect weeks 11–14 once for this teaching comparison. We still do not open the reserved test.

**Stop 5.** In fold 1, which weeks may teach the imputer its medians? Where does `model__C` point?

In [6]:
folds = []
for last_train_week, first_val_week, last_val_week in [(6,7,8),(8,9,10)]:
    tr_idx = train.index[train.week <= last_train_week].to_numpy()
    va_idx = train.index[train.week.between(first_val_week,last_val_week)].to_numpy()
    folds.append((tr_idx, va_idx))
    print(f"Fold: train weeks 1–{last_train_week} ({len(tr_idx)} rows), validate {first_val_week}–{last_val_week} ({len(va_idx)} rows)")
search = GridSearchCV(model, {"model__C": [0.1, 1.0, 10.0]}, cv=folds, scoring="balanced_accuracy", refit=True)
search.fit(X_train, y_train)
print("Mean fold balanced accuracy by C:", [(p["model__C"],round(s,3)) for p,s in zip(search.cv_results_["params"],search.cv_results_["mean_test_score"])])
print("Selected C:", search.best_params_["model__C"])
print("Refitted pipeline validation balanced accuracy:", round(balanced_accuracy_score(y_valid, search.predict(X_valid)),3))
assert all(max(train.iloc[tr].week) < min(train.iloc[va].week) for tr,va in folds)

Fold: train weeks 1–6 (120 rows), validate 7–8 (40 rows)
Fold: train weeks 1–8 (160 rows), validate 9–10 (40 rows)
Mean fold balanced accuracy by C: [(0.1, np.float64(0.912)), (1.0, np.float64(0.888)), (10.0, np.float64(0.85))]
Selected C: 0.1
Refitted pipeline validation balanced accuracy: 0.875


## 6. Apply one fitted object to later rows, including an unseen category

`technical` appears later and was absent from training. With `handle_unknown="ignore"`, the encoder keeps the learned training columns and gives zeros for that category's one-hot columns; it does **not** learn a new category during prediction. A missing numeric input receives a stored training median. This is the same operation we would call when new rows arrive.

**Stop 6.** Change only the first row's `practice_gap_s` to missing in the copy below. Will the model's stored median change? Are individual predictions guaranteed to remain the same?

In [7]:
later = X_valid.iloc[[0,1,2]].copy()
later.loc[later.index[0], "circuit_type"] = "technical"
later.loc[later.index[0], "practice_gap_s"] = float("nan")
before = search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_.copy()
print("New-row features:", later.to_string(index=False))
print("Predicted top-ten labels:", search.predict(later).tolist())
print("Stored medians still the same:", bool((before == search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()))
assert (before == search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()

New-row features:  qualifying_position  practice_gap_s  previous_top10_rate  grid_penalty circuit_type rain_forecast
                   6             NaN                0.264             0    technical            no
                  12           0.887                  NaN             0    technical            no
                  16           0.939                0.631             0    technical            no
Predicted top-ten labels: [1, 0, 0]
Stored medians still the same: True


## 7. Your change: edit one input, keep the fitted pipeline

The next cell starts with `new_qualifying_position = 20`. Before running, predict whether the top-ten label for the first later row will change. **Your pair's edit:** after the first run, set this number to **1**, rerun only this cell, and record the second outcome. Do **not** rerun `fit` or the parameter search. Explain why stored preprocessing values remain fixed even if a prediction changes.

In [8]:
new_qualifying_position = 20  # YOUR EDIT: after the first run, change 20 to 1 and rerun this cell.
scenario = later.iloc[[0]].copy()
scenario.loc[scenario.index[0], "qualifying_position"] = new_qualifying_position
print("Original qualifying position:", int(later.iloc[0].qualifying_position))
print("Scenario qualifying position:", new_qualifying_position)
print("Original / scenario predictions:", int(search.predict(later.iloc[[0]])[0]), int(search.predict(scenario)[0]))
print("Stored medians still the same:", bool((before == search.best_estimator_.named_steps["prep"].named_transformers_["num"].named_steps["fill"].statistics_).all()))

Original qualifying position: 6
Scenario qualifying position: 20
Original / scenario predictions: 1 0
Stored medians still the same: True


## 8. Diagnose the limits, then hand in

The one-line operation `search.predict(new_rows)` is useful because its preprocessing and model stay together. It does **not** select a valid time split, detect that a feature was measured after the race, or establish performance for every future circuit. `handle_unknown="ignore"` avoids an error for a new category; it does not prove good predictions on that category. The reserved test weeks 15–18 were not evaluated in this class.

**Pair discussion:** Which two checks must still be done by people? Give one possible leakage source outside the pipeline. Point to the object that contains the fitted median. Complete the worksheet and each person's exit ticket.

Reference: [scikit-learn Pipeline documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).